# Data Science: Bayesian Nonparametrics & Causal Inference

## Part 1: Dirichlet Process Mixtures for Clustering
## Part 2: Double Machine Learning for Treatment Effects

This notebook demonstrates two advanced statistical techniques that showcase rigorous applied mathematics:

1. **Dirichlet Process Mixtures**: Clustering without pre-specifying the number of clusters (infinite mixture model)
2. **Double Machine Learning**: Causal inference in high dimensions using orthogonalized scores

### References
- Ferguson, T. S. (1973). "A Bayesian Analysis of Some Nonparametric Problems"
- Sethuraman, J. (1994). "A Constructive Definition of Dirichlet Priors"
- Chernozhukov, V., et al. (2018). "Double Machine Learning for Treatment and Structural Parameters"


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import gammaln, digamma
import pandas as pd
from typing import Tuple, List, Dict
import seaborn as sns

sns.set_style('whitegrid')
np.random.seed(42)

## PART 1: BAYESIAN NONPARAMETRICS

### 1.1 Theory: Dirichlet Process

**Definition**: A Dirichlet Process $DP(\alpha, G_0)$ is a distribution over probability measures.

**Key Properties**:
- $G \sim DP(\alpha, G_0)$ means draws from $G$ are exchangeable
- $G(A) \sim \text{Dir}(\alpha G_0(A_1), \alpha G_0(A_2), \ldots)$ for any partition
- $\alpha$ = concentration parameter (higher → more uniform, lower → more clustered)
- $G_0$ = base measure (prior on component parameters)

**Stick-Breaking Construction** (Sethuraman, 1994):
$$G = \sum_{k=1}^{\infty} \pi_k \delta_{\phi_k}$$

where $\pi_k = V_k \prod_{j=1}^{k-1}(1-V_j)$ with $V_k \sim \text{Beta}(1, \alpha)$

This gives the **Chinese Restaurant Process**: Customer $n$ sits at table $k$ with probability $\propto$ (number at table $k$) or at new table with probability $\propto \alpha$


### 1.2 Dirichlet Process Mixture Model

**Generative Model**:
- $G \sim DP(\alpha, G_0)$ (draw random probability measure)
- $\phi_i | G \sim G$ (draw cluster parameters)
- $y_i | \phi_i \sim F(\cdot | \phi_i)$ (draw observations)

**Inference**: Use Gibbs sampling with Chinese Restaurant Process (CRP) prior.
- Cluster assignments: $z_i \in \{1, 2, \ldots, K_t\}$ (current number of clusters)
- Cluster parameters: $\theta_k$ (e.g., mean, variance for Gaussian mixture)


In [ ]:
class DirichletProcessGaussianMixture:
    """
    Dirichlet Process Gaussian Mixture Model using Gibbs sampling.
    
    Learns clustering without pre-specifying K.
    """
    
    def __init__(self, data: np.ndarray, alpha: float = 1.0,
                 mu0: float = 0.0, kappa0: float = 1.0,
                 nu0: float = 1.0, sigma20: float = 1.0):
        """
        Parameters:
        -----------
        data : (n, d) array
            Data matrix
        alpha : float
            DP concentration parameter
        mu0, kappa0, nu0, sigma20 : float
            Hyperparameters for Normal-Inverse-Gamma base measure
        """
        self.data = data
        self.n, self.d = data.shape
        self.alpha = alpha
        self.mu0 = mu0
        self.kappa0 = kappa0
        self.nu0 = nu0
        self.sigma20 = sigma20
        
        # Initialize: each point in its own cluster
        self.z = np.arange(self.n)
        self.K = self.n
        self.cluster_counts = np.ones(self.n)
        
    def sample_cluster_assignment(self, i: int) -> int:
        """
        Sample cluster for observation i using CRP prior.
        
        Posterior probability:
        P(z_i = k | z_{-i}, y) ∝ (n_k - δ_{z_i==k}) * p(y_i | y_{zi=k})
                                 + α * p(y_i | base measure)
        """
        # Remove point from current cluster
        current_cluster = self.z[i]
        self.cluster_counts[current_cluster] -= 1
        
        yi = self.data[i]
        
        # Get unique non-empty clusters
        unique_clusters = np.where(self.cluster_counts > 0)[0]
        
        # Log probability for each existing cluster
        log_probs = []
        for k in unique_clusters:
            cluster_data = self.data[self.z == k]
            n_k = self.cluster_counts[k]
            
            # Likelihood under cluster k
            mu_k = np.mean(cluster_data, axis=0)
            dist = np.sum((yi - mu_k) ** 2)
            likelihood = -0.5 * dist / (self.sigma20 + 0.01)
            
            # Prior (CRP)
            prior = np.log(n_k)
            log_probs.append(prior + likelihood)
        
        # Log probability for new cluster
        likelihood_new = 0.0  # Uniform over base measure
        prior_new = np.log(self.alpha)
        log_probs.append(prior_new + likelihood_new)
        
        # Softmax to get probabilities
        log_probs = np.array(log_probs)
        log_probs = log_probs - np.max(log_probs)  # Numerical stability
        probs = np.exp(log_probs) / np.sum(np.exp(log_probs))
        
        # Sample
        if len(unique_clusters) > 0:
            choice = np.random.choice(len(unique_clusters) + 1, p=probs)
            if choice < len(unique_clusters):
                new_cluster = unique_clusters[choice]
            else:
                new_cluster = np.max(self.z) + 1
        else:
            new_cluster = 0
        
        # Add point to new cluster
        if new_cluster >= len(self.cluster_counts):
            self.cluster_counts = np.append(self.cluster_counts, 0)
        
        self.z[i] = new_cluster
        self.cluster_counts[new_cluster] += 1
        self.K = len(np.unique(self.z))
        
        return new_cluster
    
    def gibbs_sample(self, n_iterations: int = 100) -> Dict:
        """
        Run Gibbs sampler.
        """
        K_trajectory = []
        
        for iteration in range(n_iterations):
            for i in range(self.n):
                self.sample_cluster_assignment(i)
            
            K_trajectory.append(self.K)
            
            if (iteration + 1) % 20 == 0:
                print(f"Iteration {iteration + 1}: K={self.K} clusters")
        
        return {
            'z': self.z,
            'K': self.K,
            'K_trajectory': K_trajectory,
            'cluster_counts': self.cluster_counts[:self.K]
        }

# Generate synthetic data: mixture of Gaussians
np.random.seed(42)
n_per_cluster = 50
true_K = 3
data_list = [
    np.random.randn(n_per_cluster, 2) + np.array([0, 0]),
    np.random.randn(n_per_cluster, 2) + np.array([5, 5]),
    np.random.randn(n_per_cluster, 2) + np.array([-4, 3])
]
data = np.vstack(data_list)
n = len(data)
true_labels = np.repeat(np.arange(true_K), n_per_cluster)

print(f"Synthetic data: {n} points, {true_K} true clusters")
print(f"Data shape: {data.shape}")

### 1.3 Inference via Gibbs Sampling

In [ ]:
# Run DP mixture model
model = DirichletProcessGaussianMixture(data, alpha=1.0)
results = model.gibbs_sample(n_iterations=150)

print(f"\nFinal number of clusters discovered: {results['K']}")
print(f"True number of clusters: {true_K}")
print(f"Cluster sizes: {results['cluster_counts']}")

### 1.4 Visualization: Clustering Results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Plot 1: Original data with true labels
ax = axes[0, 0]
colors_true = ['red', 'blue', 'green']
for k in range(true_K):
    mask = true_labels == k
    ax.scatter(data[mask, 0], data[mask, 1], label=f'Cluster {k}', alpha=0.7, s=50)
ax.set_title('True Labels (Known K=3)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: DP clustering results
ax = axes[0, 1]
for k in range(results['K']):
    mask = results['z'] == k
    ax.scatter(data[mask, 0], data[mask, 1], label=f'Cluster {k}', alpha=0.7, s=50)
ax.set_title(f'DP Clustering Results (Discovered K={results["K"]})', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: Number of clusters over iterations (mixing behavior)
ax = axes[1, 0]
ax.plot(results['K_trajectory'], linewidth=2, color='steelblue')
ax.axhline(y=true_K, color='red', linestyle='--', linewidth=2, label=f'True K={true_K}')
ax.axhline(y=results['K'], color='green', linestyle='--', linewidth=2, label=f'Final K={results["K"]}')
ax.set_xlabel('Gibbs Iteration', fontsize=11)
ax.set_ylabel('Number of Clusters (K)', fontsize=11)
ax.set_title('Convergence: Number of Clusters Over Time', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 4: Cluster size distribution
ax = axes[1, 1]
cluster_sizes = results['cluster_counts']
ax.bar(range(results['K']), cluster_sizes, alpha=0.7, color='steelblue', edgecolor='black')
ax.set_xlabel('Cluster ID', fontsize=11)
ax.set_ylabel('Cluster Size (count)', fontsize=11)
ax.set_title('Discovered Cluster Sizes', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('dp_mixture_clustering.png', dpi=150, bbox_inches='tight')
plt.show()

---

## PART 2: CAUSAL INFERENCE - DOUBLE MACHINE LEARNING

### 2.1 The Causal Inference Problem

**Goal**: Estimate treatment effect $\tau = E[Y(1) - Y(0)]$ (average treatment effect)

**Challenge**: In high dimensions ($p \gg n$), standard regression gives biased estimates

$$Y = \beta_0 + \tau T + X^T \beta + \epsilon$$

The "nuisance parameters" $\beta$ cannot be estimated accurately with LASSO when $\tau$ is small.

### 2.2 Double Machine Learning Solution

**Key Insight** (Chernozhukov et al., 2018): Use **Neyman-orthogonal scores**

1. **Residualize confounders**: 
   - $\tilde{Y} = Y - \hat{m}(X)$ (residuals after predicting Y from X)
   - $\tilde{T} = T - \hat{r}(X)$ (residuals after predicting T from X)

2. **Estimate treatment effect**:
   $$\hat{\tau} = \frac{\sum_i \tilde{T}_i \tilde{Y}_i}{\sum_i \tilde{T}_i^2}$$

3. **Why it works**: Neyman orthogonality means first-stage ML errors don't propagate to $\hat{\tau}$


In [ ]:
class DoubleMachineLearning:
    """
    Double Machine Learning for treatment effect estimation.
    
    Data generating process:
    - Y = τ*T + f_0(X) + ε_y
    - T = g_0(X) + ε_t
    - where f_0, g_0 are nonlinear functions
    """
    
    def __init__(self, Y: np.ndarray, T: np.ndarray, X: np.ndarray):
        """
        Y : (n,) outcome
        T : (n,) treatment
        X : (n, p) covariates
        """
        self.Y = Y
        self.T = T
        self.X = X
        self.n, self.p = X.shape
    
    @staticmethod
    def lasso_predict(X_train: np.ndarray, y_train: np.ndarray,
                     X_test: np.ndarray, lambda_l1: float = 0.01) -> np.ndarray:
        """
        LASSO regression prediction.
        """
        from sklearn.linear_model import LassoCV
        
        model = LassoCV(cv=5, random_state=42)
        model.fit(X_train, y_train)
        
        return model.predict(X_test)
    
    def estimate_treatment_effect(self) -> Dict:
        """
        Estimate τ using Double Machine Learning.
        """
        # Step 1: Residualize treatment
        # Predict T from X
        T_pred = self.lasso_predict(self.X, self.T, self.X)
        T_residual = self.T - T_pred
        
        # Step 2: Residualize outcome
        # Predict Y from X
        Y_pred = self.lasso_predict(self.X, self.Y, self.X)
        Y_residual = self.Y - Y_pred
        
        # Step 3: Regress residual Y on residual T
        tau_hat = np.sum(T_residual * Y_residual) / np.sum(T_residual ** 2)
        
        # Confidence interval (asymptotic)
        residuals = Y_residual - tau_hat * T_residual
        sigma2_hat = np.mean(residuals ** 2)
        var_tau = sigma2_hat / np.sum(T_residual ** 2)
        se_tau = np.sqrt(var_tau)
        ci_lower = tau_hat - 1.96 * se_tau
        ci_upper = tau_hat + 1.96 * se_tau
        
        return {
            'tau_hat': tau_hat,
            'se': se_tau,
            'ci_lower': ci_lower,
            'ci_upper': ci_upper,
            'T_residual': T_residual,
            'Y_residual': Y_residual,
            'T_pred': T_pred,
            'Y_pred': Y_pred
        }

# Generate synthetic data with nonlinear confounding
np.random.seed(42)
n = 1000
p = 100

# True parameters
tau_true = 0.5  # True treatment effect

# Generate confounders
X = np.random.randn(n, p)

# Treatment: nonlinear function of X + noise
T = np.sum(X[:, :5] ** 2, axis=1) + np.random.randn(n) * 0.5

# Outcome: treatment effect + nonlinear confounder effect + noise
Y = tau_true * T + np.sum(X[:, :5] * 2, axis=1) + np.random.randn(n) * 0.5

print(f"Sample size: n={n}")
print(f"Number of covariates: p={p}")
print(f"True treatment effect: τ={tau_true}")

### 2.3 Estimate Treatment Effect

In [ ]:
from sklearn.linear_model import LassoCV

# Estimate using DML
dml = DoubleMachineLearning(Y, T, X)
dml_results = dml.estimate_treatment_effect()

print("\nDOUBLE MACHINE LEARNING RESULTS")
print("="*70)
print(f"True Effect (τ):          {tau_true:.4f}")
print(f"DML Estimate (τ̂):         {dml_results['tau_hat']:.4f}")
print(f"Standard Error:            {dml_results['se']:.4f}")
print(f"95% CI:                    [{dml_results['ci_lower']:.4f}, {dml_results['ci_upper']:.4f}]")
print(f"Covers true effect:        {dml_results['ci_lower'] < tau_true < dml_results['ci_upper']}")
print(f"Bias:                      {dml_results['tau_hat'] - tau_true:.4f}")

### 2.4 Comparison: Naive OLS vs. DML

In [ ]:
# Compare with naive OLS
from sklearn.linear_model import LinearRegression

model_ols = LinearRegression()
model_ols.fit(np.column_stack([T, X]), Y)
tau_ols = model_ols.coef_[0]

# Compare with LASSO (penalizes confounders heavily)
model_lasso = LassoCV(cv=5, random_state=42)
model_lasso.fit(np.column_stack([T, X]), Y)
tau_lasso = model_lasso.coef_[0]

print("\nMETHOD COMPARISON")
print("="*70)
print(f"{'Method':<20} {'Estimate':>15} {'Bias':>15} {'|Error|':>15}")
print("-"*70)
print(f"{'True Effect':<20} {tau_true:>15.4f} {'':>15} {'':>15}")
print(f"{'OLS (naive)':<20} {tau_ols:>15.4f} {tau_ols - tau_true:>15.4f} {abs(tau_ols - tau_true):>15.4f}")
print(f"{'LASSO':<20} {tau_lasso:>15.4f} {tau_lasso - tau_true:>15.4f} {abs(tau_lasso - tau_true):>15.4f}")
print(f"{'Double ML':<20} {dml_results["tau_hat"]:>15.4f} {dml_results["tau_hat"] - tau_true:>15.4f} {abs(dml_results["tau_hat"] - tau_true):>15.4f}")

### 2.5 Visualization: Residual Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Residual T vs Residual Y
ax = axes[0, 0]
ax.scatter(dml_results['T_residual'], dml_results['Y_residual'], alpha=0.5, s=20)
# Add regression line
z = np.polyfit(dml_results['T_residual'], dml_results['Y_residual'], 1)
p = np.poly1d(z)
x_line = np.linspace(dml_results['T_residual'].min(), dml_results['T_residual'].max(), 100)
ax.plot(x_line, p(x_line), 'r-', linewidth=2, label=f'Slope = {dml_results["tau_hat"]:.4f}')
ax.set_xlabel('Residual Treatment', fontsize=11)
ax.set_ylabel('Residual Outcome', fontsize=11)
ax.set_title('Partialed-Out Analysis: Treatment Effect Estimation', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 2: Original T vs X (confounding)
ax = axes[0, 1]
ax.scatter(X[:, 0], T, alpha=0.5, s=20, color='steelblue')
ax.set_xlabel('First Confounder (X_1)', fontsize=11)
ax.set_ylabel('Treatment (T)', fontsize=11)
ax.set_title('Confounding: Treatment Depends on X', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

# Plot 3: Original Y vs X (confounding)
ax = axes[1, 0]
ax.scatter(X[:, 0], Y, alpha=0.5, s=20, color='darkgreen')
ax.set_xlabel('First Confounder (X_1)', fontsize=11)
ax.set_ylabel('Outcome (Y)', fontsize=11)
ax.set_title('Confounding: Outcome Depends on X', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

# Plot 4: Comparison of estimates
ax = axes[1, 1]
methods = ['True\nEffect', 'OLS\n(Naive)', 'LASSO', 'Double\nML']
estimates = [tau_true, tau_ols, tau_lasso, dml_results['tau_hat']]
colors = ['green', 'red', 'orange', 'blue']
bars = ax.bar(methods, estimates, color=colors, alpha=0.7, edgecolor='black')
ax.axhline(y=tau_true, color='green', linestyle='--', linewidth=2, alpha=0.5)
ax.set_ylabel('Treatment Effect Estimate', fontsize=11)
ax.set_title('Method Comparison: Bias in High Dimensions', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
for bar, est in zip(bars, estimates):
    ax.text(bar.get_x() + bar.get_width()/2, est, f'{est:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('dml_causal_inference.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary & Conclusions

### Dirichlet Process Mixtures
1. **Non-parametric** → No need to specify K in advance
2. **Automatic model selection** via Chinese Restaurant Process
3. **Theoretically grounded** in exchangeability and martingale limits
4. **Limitation**: Computationally expensive for large $n$; requires careful hyperparameter tuning ($\alpha$)

### Double Machine Learning
1. **High-dimensional** → Can handle $p \gg n$ without bias
2. **Robust** → Orthogonalization makes first-stage errors negligible
3. **Valid inference** → Asymptotically normal estimates with valid confidence intervals
4. **Generality** → Works for any ML estimator (Random Forests, Neural Nets, etc.)

### Key Insight
> **Rigorous statistical inference in high dimensions requires understanding the **interaction between ML estimation and causal structure**. Double ML shows that orthogonalization—removing the confounding direction—enables valid inference.** 
